# 🎓 Indic LLM Evaluation Lab: Malayalam QA with Qwen-2.5 3B & QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalvpm4321/malayalam-machine-learning/blob/main/notebooks/indic_llm_inference_server.ipynb)

**M.Tech Research Project**: Multilingual & Indic Large Language Model Evaluation and Improvement  
**Target Language**: Malayalam (`ml` / മലയാളം)  
**Base Model**: `Qwen/Qwen2.5-3B-Instruct`  
**Dataset**: AI4Bharat IndicQA Malayalam  
**Methodology**: 4-bit Quantized Low-Rank Adaptation (QLoRA, $r=16, \alpha=32$) with Assistant-Only Loss Masking  

---

### 📌 Pipeline Workflow
1. **Configuration**: Explicit flag control (`RUN_TRAINING = False`, `DEBUG_MODE = True`, `USE_QLORA_ADAPTER = True`).
2. **Data Pipeline**: Context/Article-level train/test split (80/20) with random seeding to prevent data leakage.
3. **Intelligent Context Windowing**: Truncates context *around* the ground-truth answer span to guarantee the answer and supporting evidence are never severed.
4. **Assistant-Only Loss Masking**: Prompt tokens masked with `-100` so loss is computed strictly on assistant answers.
5. **Clean Model Lifecycle**: Training wrapper is explicitly discarded and adapter reloaded cleanly for inference.
6. **True QA Token F1**: Evaluated using multiset `Counter` token overlap with Unicode NFKC preservation.
7. **Dual-Model Inference Server**: Serves verified `base` and `qlora` configurations via FastAPI and Cloudflare Quick Tunnel.

## ⚙️ SECTION 1: Configuration & Experiment Flags `[CPU / SAFE]`
Modify these parameters to control execution. For routine inference with the frontend, leave `RUN_TRAINING = False`.

In [ ]:
# =============================================================================
# EXPERIMENT & PIPELINE CONFIGURATION FLAGS
# =============================================================================
RUN_TRAINING = False          # Set to True ONLY when you explicitly want to train
FORCE_RETRAIN = False         # Set to True to overwrite existing saved adapter
RUN_EVALUATION = True         # Run test set evaluation and sanity checks
USE_QLORA_ADAPTER = True      # Load fine-tuned LoRA adapter if available
DEBUG_MODE = True             # True: Seeded debug subset (70 train / 13 test) | False: Full IndicQA

# Model & Dataset Hyperparameters
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./malayalam-qwen-qlora-chat"
DATASET_URL = "https://huggingface.co/datasets/ai4bharat/IndicQA/resolve/main/data/indicqa.ml.json"
MAX_LENGTH = 1536             # Max context + question + answer sequence length
MAX_NEW_TOKENS = 32           # Extractive QA generation limit
SEED = 42

# QLoRA Hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training Hyperparameters (Optimized for Tesla T4 ~15GB VRAM)
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3 if DEBUG_MODE else 5

print("=" * 65)
print(f"🔧 CONFIGURATION SUMMARY:")
print(f"   • Base Model:          {MODEL_ID}")
print(f"   • Run Training:        {RUN_TRAINING}")
print(f"   • Force Retrain:       {FORCE_RETRAIN}")
print(f"   • Use QLoRA Adapter:   {USE_QLORA_ADAPTER}")
print(f"   • Debug Mode:          {DEBUG_MODE}")
print(f"   • Output Directory:    {OUTPUT_DIR}")
print("=" * 65)

## 🖥️ SECTION 2: GPU Environment & Memory Check `[CPU / SAFE]`

In [ ]:
import torch
import os

def print_gpu_status(stage="Current Status"):
    print(f"\n--- [GPU Status: {stage}] ---")
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        alloc_mem = torch.cuda.memory_allocated(0) / (1024**3)
        res_mem = torch.cuda.memory_reserved(0) / (1024**3)
        print(f"GPU Name:            {device_name}")
        print(f"Total VRAM:          {total_mem:.2f} GB")
        print(f"Allocated VRAM:      {alloc_mem:.2f} GB")
        print(f"Reserved VRAM:       {res_mem:.2f} GB")
    else:
        print("⚠️ No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU!")

# Check initial hardware
!nvidia-smi
print_gpu_status("Initial Baseline")

## 📦 SECTION 3: Install Required Dependencies `[CPU / SAFE]`

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers peft accelerate bitsandbytes datasets trl sentencepiece fastapi uvicorn pydantic nest-asyncio

print("✅ All dependencies successfully installed!")

## 📊 SECTION 4: Dataset Preparation & Context-Level Splitting `[CPU / SAFE]`

- Direct JSON download from Hugging Face IndicQA Malayalam.
- **Context-level splitting (80/20)**: Groups QA pairs by paragraph context to ensure zero test leakage.
- **Seeded Random Selection**: In `DEBUG_MODE = True`, uses `random.Random(SEED).sample()` to avoid sequential ordering bias.

In [ ]:
import json
import urllib.request
import random
import numpy as np

rng = random.Random(SEED)
np.random.seed(SEED)

print(f"⬇️ Downloading IndicQA Malayalam dataset from: {DATASET_URL}...")
req = urllib.request.Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as response:
    raw_json_data = json.loads(response.read().decode("utf-8"))

# 1. Flatten into uniform records
all_records = []
for article in raw_json_data.get("data", []):
    for p_idx, paragraph in enumerate(article.get("paragraphs", [])):
        context_text = paragraph.get("context", "").strip()
        for q_idx, qa in enumerate(paragraph.get("qas", [])):
            qa_id = qa.get("id", f"indicqa_ml_{len(all_records)}")
            question_text = qa.get("question", "").strip()
            answers = qa.get("answers", [])
            answer_text = ""
            if answers and isinstance(answers, list) and len(answers) > 0:
                answer_text = answers[0].get("text", "").strip()
            
            all_records.append({
                "id": str(qa_id),
                "context": context_text,
                "question": question_text,
                "answer": answer_text,
                "context_id": f"{article.get('title', 'art')}_{p_idx}"
            })

total_raw_count = len(all_records)
print(f"Total raw QA pairs extracted: {total_raw_count}")

# 2. Quality Check & Data Cleaning
cleaned_records = []
seen_keys = set()
removed_empty_answers = 0
removed_empty_contexts = 0
removed_empty_questions = 0
removed_duplicates = 0

for r in all_records:
    if not r["answer"]:
        removed_empty_answers += 1
        continue
    if not r["context"]:
        removed_empty_contexts += 1
        continue
    if not r["question"]:
        removed_empty_questions += 1
        continue
    
    dedup_key = (r["context"], r["question"])
    if dedup_key in seen_keys:
        removed_duplicates += 1
        continue
    seen_keys.add(dedup_key)
    cleaned_records.append(r)

total_cleaned_count = len(cleaned_records)

# 3. Context-Level Splitting (80% Train, 20% Test) to prevent data leakage
unique_contexts = list(set(r["context_id"] for r in cleaned_records))
rng.shuffle(unique_contexts)

train_context_cutoff = int(0.80 * len(unique_contexts))
train_context_ids = set(unique_contexts[:train_context_cutoff])
test_context_ids = set(unique_contexts[train_context_cutoff:])

full_train_records = [r for r in cleaned_records if r["context_id"] in train_context_ids]
full_test_records = [r for r in cleaned_records if r["context_id"] in test_context_ids]

# 4. Apply Seeded Random Selection for Debug Mode
if DEBUG_MODE:
    train_records = rng.sample(full_train_records, min(70, len(full_train_records)))
    test_records = rng.sample(full_test_records, min(13, len(full_test_records)))
    split_note = f"DEBUG SUBSET (Seeded random sample: {len(train_records)} train / {len(test_records)} test)"
else:
    train_records = full_train_records
    test_records = full_test_records
    split_note = f"FULL DATASET ({len(train_records)} train / {len(test_records)} test)"

# 5. Dataset Quality Statistics
context_lengths = [len(r["context"]) for r in train_records + test_records]
answer_lengths = [len(r["answer"]) for r in train_records + test_records]

print("=" * 65)
print(f"📊 DATASET QUALITY & SPLIT REPORT ({split_note}):")
print(f"   • Total Raw Records:           {total_raw_count}")
print(f"   • Removed (Empty Answers):     {removed_empty_answers}")
print(f"   • Removed (Empty Contexts):    {removed_empty_contexts}")
print(f"   • Removed (Duplicates):        {removed_duplicates}")
print(f"   • Clean Valid Records:         {total_cleaned_count}")
print(f"   • Final Training Records:      {len(train_records)}")
print(f"   • Final Testing Records:       {len(test_records)}")
print(f"   • Avg Context Character Len:   {np.mean(context_lengths):.1f} chars (Max: {np.max(context_lengths)})")
print(f"   • Avg Answer Character Len:    {np.mean(answer_lengths):.1f} chars (Max: {np.max(answer_lengths)})")
print("=" * 65)

## 🎯 SECTION 5: Intelligent Context Windowing & Assistant-Only Loss Masking `[CPU / SAFE]`

### Essential Engineering Details:
1. **Intelligent Context Windowing (`truncate_context_around_answer`)**:
   - Rather than naively truncating tokens from the end of the sequence (which cuts off evidence if the answer is near the end), we search for the ground truth answer in the passage and construct a context window **centered around the answer**.
   - This guarantees that the answer span and surrounding supporting sentences are 100% intact.
2. **Assistant-Only Loss Masking**:
   - All prompt tokens are masked with `-100` so cross-entropy loss is computed **only on the assistant answer tokens**.

In [ ]:
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    "You are a helpful Malayalam Question Answering assistant. "
    "Answer the question using only the provided context. Give only the answer."
)

print(f"📦 Loading tokenizer for template validation: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def truncate_context_around_answer(context: str, answer: str, max_chars: int = 2500) -> str:
    """
    Intelligently preserves the answer and surrounding evidence if context exceeds length limits.
    """
    if len(context) <= max_chars:
        return context
    
    ans_idx = context.find(answer)
    if ans_idx == -1:
        # Fallback to start if not exact match
        return context[:max_chars].strip()
    
    half_window = max_chars // 2
    start_idx = max(0, ans_idx - half_window)
    end_idx = min(len(context), start_idx + max_chars)
    
    if end_idx - start_idx < max_chars:
        start_idx = max(0, end_idx - max_chars)
        
    return context[start_idx:end_idx].strip()

def format_qwen_chat_prompt(context: str, question: str) -> list:
    """Standard chat message structure for both training and inference."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
    ]

def prepare_training_sample(example, tokenizer, max_length=MAX_LENGTH):
    """
    Constructs input_ids and labels with assistant-only loss masking (-100 for prompt tokens).
    Guarantees the answer is never truncated.
    """
    raw_context = example["context"]
    question = example["question"]
    answer = example["answer"]

    # 1. Window context around the answer to prevent severing supporting evidence
    context = truncate_context_around_answer(raw_context, answer, max_chars=2500)

    # 2. Tokenize answer first to reserve space
    answer_text = f"{answer}{tokenizer.eos_token}"
    answer_ids = tokenizer.encode(answer_text, add_special_tokens=False)
    
    # 3. Format prompt with generation prompt
    prompt_messages = format_qwen_chat_prompt(context, question)
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    
    # 4. Truncate prompt from left if total exceeds max_length, keeping full answer
    total_len = len(prompt_ids) + len(answer_ids)
    if total_len > max_length:
        allowed_prompt_len = max_length - len(answer_ids)
        prompt_ids = prompt_ids[:allowed_prompt_len]
    
    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)
    labels = ([-100] * len(prompt_ids)) + answer_ids
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_len": len(prompt_ids),
        "answer_len": len(answer_ids)
    }

# Validation on a sample from the training set
sample_check = prepare_training_sample(train_records[0], tokenizer)
decoded_answer_from_labels = tokenizer.decode(
    [token for token in sample_check["labels"] if token != -100],
    skip_special_tokens=True
).strip()

print("=" * 65)
print("🔍 ASSISTANT-ONLY LOSS MASKING VERIFICATION:")
print(f"   • Total Tokens:               {len(sample_check['input_ids'])}")
print(f"   • Prompt Tokens (Masked -100):{sample_check['prompt_len']}")
print(f"   • Answer Tokens (Active Loss):{sample_check['answer_len']}")
print(f"   • Original Dataset Answer:    '{train_records[0]['answer']}'")
print(f"   • Decoded Loss Tokens:        '{decoded_answer_from_labels}'")
print("=" * 65)

assert sample_check["answer_len"] > 0, "FATAL: Answer token length is 0!"
assert decoded_answer_from_labels == train_records[0]["answer"].strip(), "FATAL: Decoded answer mismatch!"
print("✅ Masking verified: Loss will be computed strictly on assistant answers!")

## 🤖 SECTION 6: Base Model Loading (4-Bit NF4 Quantization) `[⚠️ GPU / EXPENSIVE]`
We load `Qwen/Qwen2.5-3B-Instruct` in 4-bit NormalFloat4 precision with double quantization for low memory footprint (~2.5 GB VRAM).

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import gc

print_gpu_status("Before Model Loading")
print(f"📦 Loading {MODEL_ID} in 4-bit NF4 precision...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.config.use_cache = False

print_gpu_status("After 4-bit Base Model Loaded")
print("✅ Base Qwen 2.5 3B model loaded successfully on GPU!")

## 🚀 SECTION 7: QLoRA Fine-Tuning Pipeline `[⚠️ GPU / EXPENSIVE]`

- Executes **only when `RUN_TRAINING = True`**.
- **Clean Model Lifecycle**: The training wrapper is discarded after saving the adapter, ensuring inference is always performed from a pristine model state.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from datasets import Dataset
import time

adapter_exists = os.path.exists(os.path.join(OUTPUT_DIR, "adapter_model.safetensors")) or \
                 os.path.exists(os.path.join(OUTPUT_DIR, "adapter_model.bin"))

if RUN_TRAINING and (not adapter_exists or FORCE_RETRAIN):
    print("=" * 65)
    print("🚀 STARTING QLORA FINE-TUNING PIPELINE")
    print("=" * 65)
    
    # 1. Prepare base model for k-bit training
    peft_train_model = prepare_model_for_kbit_training(base_model)
    
    # 2. Configure LoRA parameters
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_train_model = get_peft_model(peft_train_model, lora_config)
    
    # 3. Print Trainable Parameters
    trainable_params, all_params = peft_train_model.get_nb_trainable_parameters()
    trainable_pct = (trainable_params / all_params) * 100
    print(f"📊 PARAMETER STATISTICS:")
    print(f"   • Trainable Parameters:    {trainable_params:,} ({trainable_params/1e6:.2f} M)")
    print(f"   • Total Parameters:        {all_params:,} ({all_params/1e9:.2f} B)")
    print(f"   • Trainable Percentage:    {trainable_pct:.4f} %")
    
    # 4. Prepare Tokenized Hugging Face Dataset
    tokenized_train_data = [prepare_training_sample(r, tokenizer) for r in train_records]
    hf_train_dataset = Dataset.from_list(tokenized_train_data)
    
    # 5. Training Arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        fp16=True,
        logging_steps=5,
        save_strategy="no",
        optim="paged_adamw_8bit",
        report_to="none",
        seed=SEED
    )
    
    trainer = Trainer(
        model=peft_train_model,
        args=training_args,
        train_dataset=hf_train_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True)
    )
    
    start_train_time = time.time()
    train_result = trainer.train()
    total_train_duration = time.time() - start_train_time
    
    print(f"\n✅ Training Complete in {total_train_duration:.2f} seconds! (Final Loss: {train_result.training_loss:.4f})")
    
    # 6. Save Adapter and Tokenizer
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    peft_train_model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"💾 Saved fine-tuned QLoRA adapter to: {OUTPUT_DIR}")
    
    # 7. Clean up training wrapper memory
    del peft_train_model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    print_gpu_status("After Training Cleanup")
elif adapter_exists:
    print(f"ℹ️ Saved QLoRA adapter found at '{OUTPUT_DIR}'. Training skipped (RUN_TRAINING = {RUN_TRAINING}).")
else:
    print(f"ℹ️ RUN_TRAINING is set to False and no adapter exists yet. Using Base Model.")

## 🔌 SECTION 8: QLoRA Adapter Loading & Dual-Config Model Setup `[⚠️ GPU / EXPENSIVE]`

We load the saved adapter onto the base model and provide a zero-overhead context manager (`model_inference_context`) so API requests can switch between `base` and `qlora` without reloading weights.

In [ ]:
from peft import PeftModel
from contextlib import contextmanager

active_model = base_model
qlora_loaded = False

adapter_file_present = os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json"))

if USE_QLORA_ADAPTER and adapter_file_present:
    print(f"📦 Loading saved QLoRA adapter from '{OUTPUT_DIR}' onto base model...")
    try:
        active_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
        qlora_loaded = True
        print("✅ QLoRA adapter loaded and ready for inference!")
    except Exception as e:
        print(f"⚠️ Error loading adapter: {e}. Falling back to base model.")
        active_model = base_model
else:
    print("ℹ️ Running with Base Model (QLoRA adapter not attached).")

@contextmanager
def model_inference_context(config_name: str):
    """
    Zero-overhead switcher between Base model and QLoRA adapter in memory.
    """
    if config_name.lower() == "base" and qlora_loaded:
        with active_model.disable_adapter():
            yield active_model
    else:
        yield active_model

print_gpu_status("Dual-Model Inference Ready")

## 🧪 SECTION 9: Model Sanity Checks & Test Set Evaluation `[⚠️ GPU / EXPENSIVE]`

- **True QA F1**: Calculated using `collections.Counter` multiset token overlap.
- **Explicit Mode Reporting**: Reports separate, transparent results for `Base` and `QLoRA`.

In [ ]:
import re
import string
import unicodedata
from collections import Counter

def normalize_indic_answer(text: str) -> str:
    """Normalizes whitespace, Unicode NFKC, and punctuation for Malayalam QA evaluation."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    # Remove surrounding punctuation
    text = text.strip().strip(string.punctuation + "।॥?,.!;:-")
    # Standardize whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compute_qa_metrics(prediction: str, ground_truth: str) -> dict:
    """
    Computes Exact Match (EM) and True Token-level F1 score using Counter multiset overlap.
    """
    norm_pred = normalize_indic_answer(prediction)
    norm_gold = normalize_indic_answer(ground_truth)
    
    if not norm_gold:
        return {"exact_match": None, "f1": None}
    
    # Exact Match
    em = 1 if norm_pred == norm_gold else 0
    
    # Token-level Multiset F1
    pred_tokens = norm_pred.split()
    gold_tokens = norm_gold.split()
    
    if not pred_tokens or not gold_tokens:
        f1 = 1.0 if norm_pred == norm_gold else 0.0
        return {"exact_match": em, "f1": f1}
    
    # True token-level overlap via Counter multiset intersection
    common = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if common == 0:
        return {"exact_match": em, "f1": 0.0}
    
    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    return {"exact_match": em, "f1": round(f1, 4)}

def run_single_inference(context: str, question: str, config: str = "base") -> str:
    messages = format_qwen_chat_prompt(context, question)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(base_model.device)
    
    with torch.inference_mode():
        with model_inference_context(config) as m:
            outputs = m.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

if RUN_EVALUATION:
    active_eval_mode = "QLoRA (Fine-Tuned)" if qlora_loaded else "Base Model (Zero-Shot Baseline)"
    print("=" * 65)
    print(f"🧪 1. MANUAL MALAYALAM SANITY TESTS [Testing: {active_eval_mode}]:")
    
    sanity_1 = run_single_inference(
        "കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്. കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ ഭാഗത്താണ് സ്ഥിതി ചെയ്യുന്നത്.",
        "കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?",
        config="qlora" if qlora_loaded else "base"
    )
    print(f"   • Test 1 (Trivandrum)   Expected: 'തിരുവനന്തപുരം' | Predicted: '{sanity_1}'")
    
    sanity_2 = run_single_inference(
        "ഇന്ത്യയുടെ ദേശീയ പതാകയിൽ മൂന്ന് നിറങ്ങളുണ്ട്. വെള്ള നിറത്തിന്റെ മധ്യഭാഗത്ത് നീല നിറത്തിലുള്ള അശോകചക്രം ഉണ്ട്.",
        "ഇന്ത്യയുടെ ദേശീയ പതാകയുടെ മധ്യഭാഗത്ത് എന്താണ് ഉള്ളത്?",
        config="qlora" if qlora_loaded else "base"
    )
    print(f"   • Test 2 (Ashoka Wheel)  Expected: 'അശോകചക്രം'       | Predicted: '{sanity_2}'")
    
    # Overfitting Sanity Test on 3 Training Samples
    print(f"\n🧪 2. TRAINING-SET OVERFITTING SANITY CHECK (3 Samples):")
    for i, item in enumerate(train_records[:3]):
        pred = run_single_inference(item["context"], item["question"], config="qlora" if qlora_loaded else "base")
        metrics = compute_qa_metrics(pred, item["answer"])
        print(f"   [{i+1}] Expected: '{item['answer']}' | Pred: '{pred}' | EM: {metrics['exact_match']} | F1: {metrics['f1']}")
    
    # Test-Set Evaluation
    print(f"\n🧪 3. TEST-SET EVALUATION ({len(test_records)} Samples):")
    test_em_total = 0
    test_f1_total = 0
    latencies = []
    
    for item in test_records:
        t0 = time.time()
        pred = run_single_inference(item["context"], item["question"], config="qlora" if qlora_loaded else "base")
        latencies.append((time.time() - t0) * 1000)
        metrics = compute_qa_metrics(pred, item["answer"])
        test_em_total += (metrics["exact_match"] or 0)
        test_f1_total += (metrics["f1"] or 0.0)
    
    em_score = (test_em_total / len(test_records)) * 100
    f1_score = (test_f1_total / len(test_records)) * 100
    avg_latency = np.mean(latencies)
    
    print("=" * 65)
    print(f"📊 EVALUATION RESULTS SUMMARY ({active_eval_mode}):")
    print(f"   • Exact Match (EM):     {em_score:.2f} %")
    print(f"   • Token F1 Score:       {f1_score:.2f} %")
    print(f"   • Avg Latency:          {avg_latency:.1f} ms")
    print("=" * 65)
    
    # Save Experiment Summary for Thesis
    exp_summary = {
        "model": MODEL_ID,
        "configuration": "qlora" if qlora_loaded else "base",
        "dataset": "IndicQA Malayalam",
        "train_samples": len(train_records),
        "test_samples": len(test_records),
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "exact_match_pct": round(em_score, 2),
        "f1_pct": round(f1_score, 2),
        "avg_latency_ms": round(avg_latency, 1),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }
    with open("experiment_summary.json", "w", encoding="utf-8") as f:
        json.dump(exp_summary, f, indent=2, ensure_ascii=False)
    print("📄 Saved experiment_summary.json for M.Tech research records!")

## 🧪 SECTION 10: Model & Pipeline Validation Tests `[MODEL CHECK / GPU]`

In [ ]:
def run_automated_validation_tests():
    print("=" * 65)
    print("🔍 RUNNING AUTOMATED PIPELINE & MODEL VALIDATION TESTS:")
    print("=" * 65)
    tests = []
    
    # Test A: Dataset Loaded
    t_a = len(train_records) > 0 and len(test_records) > 0
    tests.append(("test_dataset_loaded", t_a))
    
    # Test B: No Empty Answers
    t_b = all(bool(r["answer"].strip()) for r in train_records + test_records)
    tests.append(("test_no_empty_answers", t_b))
    
    # Test C: Chat Template Formatting
    t_c = len(format_qwen_chat_prompt("ക", "ച")) == 2
    tests.append(("test_chat_template", t_c))
    
    # Test D: Assistant-Only Labels Masking
    sample_tok = prepare_training_sample(train_records[0], tokenizer)
    t_d = sample_tok["answer_len"] > 0 and (sample_tok["labels"][0] == -100)
    tests.append(("test_answer_labels_masking", t_d))
    
    # Test E: Model Loaded on CUDA
    t_e = next(base_model.parameters()).is_cuda
    tests.append(("test_model_loaded_on_cuda", t_e))
    
    # Test F: True Multiset F1 Logic
    metrics_test = compute_qa_metrics("തിരുവനന്തപുരം", "തിരുവനന്തപുരം")
    t_f = metrics_test["exact_match"] == 1 and metrics_test["f1"] == 1.0
    tests.append(("test_true_multiset_f1_computation", t_f))
    
    all_passed = True
    for name, status in tests:
        status_str = "✅ PASS" if status else "❌ FAIL"
        if not status:
            all_passed = False
        print(f"   {status_str} : {name}")
    print("=" * 65)
    return all_passed

run_automated_validation_tests()

## 🌐 SECTION 11: FastAPI REST Inference Server `[CPU / SAFE]`

- `GET /health` reports active GPU stats and verified configurations: `["base", "qlora"]`.
- `POST /predict` accepts `configuration = "base"` or `configuration = "qlora"`.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import time

app = FastAPI(title="Indic LLM Evaluation Server", version="2.1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictionRequest(BaseModel):
    language: str = "ml"
    task: str = "qa"
    configuration: str = "qlora"  # "base" | "qlora"
    context: str
    question: str
    expected_answer: Optional[str] = ""

@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "supported_configurations": ["base", "qlora"],
        "active_configuration": "qlora" if qlora_loaded else "base",
        "qlora_adapter_loaded": qlora_loaded,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "gpu_memory_used_gb": round(torch.cuda.memory_allocated(0) / (1024**3), 2) if torch.cuda.is_available() else 0,
        "version": "2.1.0"
    }

@app.post("/predict")
def predict(req: PredictionRequest):
    t0 = time.time()
    config_choice = req.configuration.lower() if req.configuration else "base"
    
    # Format prompt
    messages = format_qwen_chat_prompt(req.context, req.question)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(base_model.device)
    
    with torch.inference_mode():
        with model_inference_context(config_choice) as m:
            outputs = m.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
    
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    generated_answer = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    latency_ms = int((time.time() - t0) * 1000)
    
    # Compute Evaluation Metrics
    metrics = compute_qa_metrics(generated_answer, req.expected_answer or "")
    
    return {
        "answer": generated_answer,
        "expected_answer": req.expected_answer or None,
        "exact_match": metrics["exact_match"],
        "f1": metrics["f1"],
        "similarity": metrics["f1"] if metrics["f1"] is not None else (1.0 if metrics["exact_match"] == 1 else 0.0),
        "latency_ms": latency_ms,
        "model": MODEL_ID,
        "configuration": config_choice,
        "language": req.language
    }

print("✅ FastAPI endpoints created: GET /health, POST /predict")

## 🚀 SECTION 12: Cloudflare Quick Tunnel & Server Startup `[CPU / SAFE]`

Starts the Cloudflare tunnel in the background, prints the **active public URL**, and runs Uvicorn in a daemon thread so Colab remains non-blocking.

In [ ]:
import os
import subprocess
import time
import uvicorn
import threading
import re

# 1. Clean up any existing cloudflared processes
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

# 2. Download Cloudflare tunnel binary if missing
if not os.path.exists("cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# 3. Launch Cloudflare tunnel
log_file = open("tunnel.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=log_file, stderr=log_file)

# 4. Poll tunnel.log until URL appears
tunnel_url = None
print("⏳ Establishing Cloudflare Tunnel connection...")
for _ in range(30):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
            if matches:
                tunnel_url = matches[-1]
                break

# 5. Start Uvicorn in background thread
def start_uvicorn():
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()

print("=" * 65)
if tunnel_url:
    print("🎉 YOUR ACTIVE PUBLIC TUNNEL URL IS:")
    print(f"👉 {tunnel_url}")
    print("\nCopy this URL and paste it into your local Web App Settings page!")
else:
    print("⚠️ Tunnel started. Check tunnel.log if the URL didn't print within 30s.")
print("=" * 65)